In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="03-google-street-view",
    notebook_name="03_evaluation_and_serving.ipynb"
)

# Evaluation and Serving -- From IoU to Billions of Images

## What This Notebook Covers

You have built an awesome object detection model (Notebook 02). Now the big questions are:
1. **How do you know if it is any good?** (Evaluation)
2. **How do you run it on billions of Street View images?** (Serving)

Think of it like baking a cake. Building the model was mixing the ingredients. This notebook is about:
- Tasting the cake (evaluation metrics)
- Running a bakery that serves 1,000 cakes per hour (production serving)

We will cover:
1. IoU (Intersection over Union) -- the foundation of detection evaluation
2. Precision at different IoU thresholds
3. Average Precision (AP) -- step by step
4. Mean Average Precision (mAP) -- the gold standard
5. Precision-Recall curves
6. Non-Maximum Suppression (NMS) -- step by step with visualization
7. Batch prediction pipeline architecture
8. Hard negative mining
9. User feedback loop
10. Scaling to billions of images

---

## Staff-Level Technical Summary

Object detection evaluation centers on IoU-based matching between predicted and ground truth boxes, with precision computed at various IoU thresholds, aggregated into Average Precision (AP) per class and Mean Average Precision (mAP) across classes. The serving pipeline separates CPU-bound preprocessing from GPU-bound inference for independent scaling, uses NMS for post-processing, and incorporates a feedback loop with hard negative mining for continuous improvement. For Street View, we optimize the confidence threshold for high recall (catching every face) even at the cost of some precision (occasional false blurs).

## 1. IoU (Intersection over Union)

### The 12-Year-Old Version

Imagine you and your friend both draw a rectangle around a face in a photo. IoU answers: "How much do your rectangles agree?"

- **Intersection**: The area where BOTH rectangles overlap (like the shared part of a Venn diagram)
- **Union**: The TOTAL area covered by either rectangle
- **IoU** = Intersection / Union

If your rectangles are identical: IoU = 1.0 (perfect!)
If they do not overlap at all: IoU = 0.0 (terrible!)

### Why IoU Matters

IoU is the **referee** of object detection. It decides whether a prediction is "correct" (True Positive) or "wrong" (False Positive). A common threshold is IoU >= 0.5: if your predicted box overlaps at least 50% with the ground truth, you get a point.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === IoU Implementation ===

def compute_iou(box1, box2):
    """
    Compute Intersection over Union between two bounding boxes.
    
    Each box is [x, y, width, height] where (x, y) is the top-left corner.
    
    12-year-old version:
    1. Find the rectangle where both boxes overlap
    2. Calculate that overlap area
    3. Calculate the total area covered by either box
    4. Divide overlap by total = IoU
    
    Staff-level detail:
    Time complexity: O(1)
    For batched computation over N boxes, use vectorized operations: O(N)
    """
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    # Convert to (x_min, y_min, x_max, y_max)
    x1_min, y1_min, x1_max, y1_max = x1, y1, x1 + w1, y1 + h1
    x2_min, y2_min, x2_max, y2_max = x2, y2, x2 + w2, y2 + h2
    
    # Intersection rectangle
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    # Intersection area (0 if no overlap)
    inter_width = max(0, inter_x_max - inter_x_min)
    inter_height = max(0, inter_y_max - inter_y_min)
    intersection = inter_width * inter_height
    
    # Union area
    area1 = w1 * h1
    area2 = w2 * h2
    union = area1 + area2 - intersection  # Subtract intersection to avoid double counting!
    
    if union == 0:
        return 0.0
    
    return intersection / union


# === Visualize IoU with Different Overlap Scenarios ===

scenarios = [
    {"name": "Perfect Match\nIoU = 1.00", "gt": [50, 50, 100, 80], "pred": [50, 50, 100, 80]},
    {"name": "Great Overlap\nIoU = 0.68", "gt": [50, 50, 100, 80], "pred": [60, 55, 100, 80]},
    {"name": "Decent Overlap\nIoU = 0.33", "gt": [50, 50, 100, 80], "pred": [90, 60, 100, 80]},
    {"name": "Poor Overlap\nIoU = 0.14", "gt": [50, 50, 100, 80], "pred": [120, 70, 100, 80]},
    {"name": "No Overlap\nIoU = 0.00", "gt": [50, 50, 100, 80], "pred": [250, 50, 100, 80]},
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, scenario in zip(axes, scenarios):
    gt = scenario["gt"]
    pred = scenario["pred"]
    iou = compute_iou(gt, pred)
    
    ax.set_xlim(0, 400)
    ax.set_ylim(0, 200)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    
    # Draw ground truth (green)
    gt_rect = patches.Rectangle((gt[0], gt[1]), gt[2], gt[3],
                                 linewidth=2.5, edgecolor='green', facecolor='green', alpha=0.25,
                                 label='Ground Truth')
    ax.add_patch(gt_rect)
    
    # Draw prediction (red)
    pred_rect = patches.Rectangle((pred[0], pred[1]), pred[2], pred[3],
                                   linewidth=2.5, edgecolor='red', facecolor='red', alpha=0.25,
                                   label='Prediction')
    ax.add_patch(pred_rect)
    
    # Color based on IoU threshold
    color = 'green' if iou >= 0.5 else 'red'
    title = scenario['name'] if '=' in scenario['name'] else f"{scenario['name']}\nIoU = {iou:.2f}"
    ax.set_title(title, fontsize=11, fontweight='bold', color=color)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Intersection over Union (IoU) -- The Foundation of Detection Evaluation',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

# Verify with computed values
print("IoU values computed:")
for s in scenarios:
    iou = compute_iou(s['gt'], s['pred'])
    status = 'TP (correct)' if iou >= 0.5 else 'FP (incorrect)'
    print(f"  {s['name'].split(chr(10))[0]:>20}: IoU={iou:.4f}  -> {status} at threshold 0.5")

In [ ]:
# === Detailed IoU Calculation Step by Step ===

def compute_iou_detailed(box1, box2):
    """
    Same as compute_iou but prints every step.
    Perfect for understanding and for interviews.
    """
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    print("=== Step-by-Step IoU Computation ===")
    print(f"  Box 1 (Ground Truth): x={x1}, y={y1}, w={w1}, h={h1}")
    print(f"  Box 2 (Prediction):   x={x2}, y={y2}, w={w2}, h={h2}")
    print()
    
    # Step 1: Convert to corner format
    x1_min, y1_min, x1_max, y1_max = x1, y1, x1 + w1, y1 + h1
    x2_min, y2_min, x2_max, y2_max = x2, y2, x2 + w2, y2 + h2
    print("Step 1: Convert to corner format")
    print(f"  Box 1: ({x1_min}, {y1_min}) to ({x1_max}, {y1_max})")
    print(f"  Box 2: ({x2_min}, {y2_min}) to ({x2_max}, {y2_max})")
    print()
    
    # Step 2: Find intersection
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    print("Step 2: Find intersection rectangle")
    print(f"  x_range: max({x1_min}, {x2_min})={inter_x_min} to min({x1_max}, {x2_max})={inter_x_max}")
    print(f"  y_range: max({y1_min}, {y2_min})={inter_y_min} to min({y1_max}, {y2_max})={inter_y_max}")
    
    inter_w = max(0, inter_x_max - inter_x_min)
    inter_h = max(0, inter_y_max - inter_y_min)
    intersection = inter_w * inter_h
    print(f"  Intersection: {inter_w} x {inter_h} = {intersection} pixels^2")
    print()
    
    # Step 3: Compute union
    area1 = w1 * h1
    area2 = w2 * h2
    union = area1 + area2 - intersection
    print("Step 3: Compute union")
    print(f"  Area 1 = {w1} x {h1} = {area1}")
    print(f"  Area 2 = {w2} x {h2} = {area2}")
    print(f"  Union = {area1} + {area2} - {intersection} = {union}")
    print()
    
    # Step 4: Divide
    iou = intersection / union if union > 0 else 0.0
    print(f"Step 4: IoU = intersection / union = {intersection} / {union} = {iou:.4f}")
    return iou


# Run detailed computation
gt_box = [50, 50, 100, 80]
pred_box = [70, 60, 100, 80]
iou = compute_iou_detailed(gt_box, pred_box)

## 2. Precision at Different IoU Thresholds

### The 12-Year-Old Version

Imagine your object detector draws 6 bounding boxes on an image, but there are only 2 real faces. We need to figure out: of those 6 boxes, how many are actually correct?

But "correct" depends on how strict we are! If we say "anything that overlaps a little is correct" (IoU > 0.1), we will be generous. If we say "it has to be nearly perfect" (IoU > 0.9), we will be very strict.

```
Precision = Correct Detections / Total Detections
```

This is exactly the example from the PDF: same detections, different thresholds, different precision scores.

In [ ]:
# === Precision at Different IoU Thresholds ===
# (Directly matching the PDF example with 6 detections and 2 ground truths)

# Ground truth: 2 real faces
ground_truths = [
    [50, 50, 80, 100],   # Face 1
    [250, 60, 70, 90],   # Face 2
]

# Model detections: 6 bounding boxes (some good, some bad)
detections = [
    {"bbox": [52, 48, 78, 102], "conf": 0.95, "iou_with_gt": None},   # Very close to GT[0]
    {"bbox": [55, 55, 75, 95],  "conf": 0.88, "iou_with_gt": None},   # Close to GT[0]
    {"bbox": [248, 58, 72, 92], "conf": 0.85, "iou_with_gt": None},   # Very close to GT[1]
    {"bbox": [80, 100, 60, 70], "conf": 0.75, "iou_with_gt": None},   # Partial overlap with GT[0]
    {"bbox": [300, 150, 50, 50],"conf": 0.60, "iou_with_gt": None},   # No overlap (false positive)
    {"bbox": [180, 180, 40, 40],"conf": 0.40, "iou_with_gt": None},   # No overlap (false positive)
]

# Compute IoU of each detection with its best-matching ground truth
for det in detections:
    best_iou = 0
    for gt in ground_truths:
        iou = compute_iou(det['bbox'], gt)
        best_iou = max(best_iou, iou)
    det['iou_with_gt'] = best_iou

# Show the IoU values
print("=== Detection IoU Values ===")
print(f"{'Detection #':>12} {'Confidence':>12} {'Best IoU':>10} {'Bbox':>25}")
print("-" * 65)
for i, det in enumerate(detections):
    print(f"{i+1:>12} {det['conf']:>12.2f} {det['iou_with_gt']:>10.4f} {str(det['bbox']):>25}")

# Compute precision at different thresholds
print("\n=== Precision at Different IoU Thresholds ===")
print(f"{'IoU Threshold':>15} {'Correct':>10} {'Total':>8} {'Precision':>12}")
print("-" * 50)

thresholds = [0.7, 0.5, 0.3, 0.1]
precisions_at_threshold = []
for threshold in thresholds:
    correct = sum(1 for d in detections if d['iou_with_gt'] >= threshold)
    total = len(detections)
    precision = correct / total
    precisions_at_threshold.append(precision)
    print(f"{threshold:>15.1f} {correct:>10} {total:>8} {precision:>12.4f}")

print("\nProblem: Precision changes with the IoU threshold!")
print("Which threshold should we report? There is no single right answer.")
print("Solution: Average Precision (AP) averages across thresholds.")

In [ ]:
# === Visualize the detections and their IoU values ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Create a synthetic image
img = np.zeros((300, 400, 3), dtype=np.uint8)
img[:120, :] = [135, 206, 235]
img[120:, :] = [100, 150, 100]
img[50:150, 50:130] = [220, 180, 150]  # Face 1
img[60:150, 250:320] = [200, 165, 135]  # Face 2

for threshold, ax in zip([0.7, 0.5, 0.1], axes):
    ax.imshow(img)
    
    # Draw ground truths
    for gt in ground_truths:
        rect = patches.Rectangle((gt[0], gt[1]), gt[2], gt[3],
                                  linewidth=2, edgecolor='green', facecolor='none', linestyle='--')
        ax.add_patch(rect)
    
    # Draw detections, colored by correctness at this threshold
    correct = 0
    for det in detections:
        is_correct = det['iou_with_gt'] >= threshold
        color = '#2ecc71' if is_correct else '#e74c3c'
        label = 'TP' if is_correct else 'FP'
        if is_correct:
            correct += 1
        
        x, y, w, h = det['bbox']
        rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x+w, y, label, color=color, fontsize=8, fontweight='bold')
    
    precision = correct / len(detections)
    ax.set_title(f'IoU Threshold = {threshold}\nPrecision = {correct}/{len(detections)} = {precision:.2f}',
                 fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Same Detections, Different Thresholds = Different Precision\n(Green dashed = Ground Truth, Solid = Predictions)',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 3. Average Precision (AP) -- Step by Step

### The 12-Year-Old Version

AP solves the "which threshold?" problem. Instead of picking one threshold, we look at precision across ALL thresholds and take the average. It is like grading a student on ALL their tests, not just one.

Here is the step-by-step recipe:
1. Sort all detections by confidence (most confident first)
2. Go through them one by one. For each:
   - Check if it matches a ground truth box (IoU >= threshold)
   - Track running precision and recall
3. Plot the precision-recall curve
4. AP = area under the precision-recall curve

### Staff-Level Detail

There are different AP computation methods:
- **PASCAL VOC 2008**: 11-point interpolation (sample at recall = 0.0, 0.1, ..., 1.0)
- **PASCAL VOC 2010+**: All-points interpolation (more accurate)
- **COCO**: AP averaged across IoU thresholds 0.5:0.05:0.95

In [ ]:
# === Average Precision: Complete Step-by-Step Computation ===

def compute_ap_step_by_step(detections, ground_truths, iou_threshold=0.5, verbose=True):
    """
    Compute Average Precision with detailed output.
    
    12-year-old version:
    1. Sort guesses from most confident to least confident
    2. Go through each guess and check: 'Was this right?'
    3. Keep a running score of precision and recall
    4. Average the precision values = AP
    
    Staff-level detail:
    This implements the 11-point interpolation method (PASCAL VOC 2008).
    For each of 11 recall levels, we take the maximum precision
    at that recall or higher. This smooths the PR curve to be
    monotonically decreasing.
    """
    # Sort by confidence (highest first)
    sorted_dets = sorted(detections, key=lambda d: d['conf'], reverse=True)
    
    num_gt = len(ground_truths)
    matched_gt = set()
    
    precisions = []
    recalls = []
    tp_count = 0
    fp_count = 0
    
    if verbose:
        print(f"=== Average Precision Computation (IoU threshold = {iou_threshold}) ===")
        print(f"Ground truth boxes: {num_gt}")
        print(f"Total detections: {len(sorted_dets)}")
        print()
        print(f"{'Step':>4} {'Conf':>6} {'IoU':>6} {'Match':>6} {'TP':>4} {'FP':>4} {'Prec':>8} {'Recall':>8}")
        print("-" * 55)
    
    for i, det in enumerate(sorted_dets):
        # Find best matching ground truth
        best_iou = 0
        best_gt_idx = -1
        for gt_idx, gt in enumerate(ground_truths):
            if gt_idx in matched_gt:
                continue
            iou = compute_iou(det['bbox'], gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        # Is it a TP or FP?
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp_count += 1
            matched_gt.add(best_gt_idx)
            match_str = "TP"
        else:
            fp_count += 1
            match_str = "FP"
        
        precision = tp_count / (tp_count + fp_count)
        recall = tp_count / num_gt
        precisions.append(precision)
        recalls.append(recall)
        
        if verbose:
            print(f"{i+1:>4} {det['conf']:>6.2f} {best_iou:>6.2f} {match_str:>6} "
                  f"{tp_count:>4} {fp_count:>4} {precision:>8.4f} {recall:>8.4f}")
    
    # Compute AP using 11-point interpolation
    ap = 0.0
    if verbose:
        print(f"\n--- 11-Point Interpolation ---")
    for r_thresh in np.arange(0, 1.1, 0.1):
        max_prec = 0
        for p, r in zip(precisions, recalls):
            if r >= r_thresh:
                max_prec = max(max_prec, p)
        ap += max_prec
        if verbose:
            print(f"  Recall >= {r_thresh:.1f}: max precision = {max_prec:.4f}")
    
    ap /= 11.0
    if verbose:
        print(f"\n  AP = sum / 11 = {ap:.4f}")
    
    return ap, precisions, recalls


# === Example: Face Detection AP ===

face_ground_truths = [
    [50, 50, 60, 80],     # Face 1
    [200, 100, 50, 70],   # Face 2
    [350, 60, 55, 75],    # Face 3
]

face_detections = [
    {"bbox": [52, 48, 58, 82],  "conf": 0.95},  # Good match with Face 1
    {"bbox": [198, 98, 52, 72], "conf": 0.90},  # Good match with Face 2
    {"bbox": [400, 200, 40, 50],"conf": 0.85},  # False positive
    {"bbox": [345, 55, 60, 80], "conf": 0.70},  # Good match with Face 3
    {"bbox": [55, 55, 55, 75],  "conf": 0.60},  # Duplicate of Face 1 (already matched)
    {"bbox": [150, 250, 30, 30],"conf": 0.40},  # False positive
]

ap_face, prec_face, rec_face = compute_ap_step_by_step(
    face_detections, face_ground_truths, iou_threshold=0.5
)

## 4. mAP (Mean Average Precision) and PR Curves

### The 12-Year-Old Version

AP tells us how good the model is at detecting ONE class (like faces). But our model detects TWO classes: faces AND license plates. mAP averages the AP across all classes:

```
mAP = (AP_face + AP_plate) / 2
```

mAP is the **gold standard** metric for object detection. When papers say "our model achieves mAP 42.0 on COCO," this is what they mean.

In [ ]:
# === Compute AP for License Plates Too ===

plate_ground_truths = [
    [100, 200, 80, 30],
    [300, 180, 75, 25],
]

plate_detections = [
    {"bbox": [102, 198, 78, 32], "conf": 0.93},  # Good match
    {"bbox": [298, 178, 77, 27], "conf": 0.87},  # Good match
    {"bbox": [450, 150, 60, 20], "conf": 0.55},  # False positive
]

print("=" * 60)
print("LICENSE PLATE DETECTION")
print("=" * 60)
ap_plate, prec_plate, rec_plate = compute_ap_step_by_step(
    plate_detections, plate_ground_truths, iou_threshold=0.5
)

# === Compute mAP ===
mAP = (ap_face + ap_plate) / 2

print("\n" + "=" * 60)
print("MEAN AVERAGE PRECISION (mAP)")
print("=" * 60)
print(f"  AP (face):          {ap_face:.4f}")
print(f"  AP (license plate): {ap_plate:.4f}")
print(f"  mAP = ({ap_face:.4f} + {ap_plate:.4f}) / 2 = {mAP:.4f}")
print()
print("mAP is THE metric you report in interviews and papers.")
print("Higher is better. State-of-the-art on COCO is ~60+ mAP.")

In [ ]:
# === Precision-Recall Curves ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Face PR curve
ax = axes[0]
ax.step(rec_face, prec_face, 'r-', linewidth=2.5, where='post')
ax.fill_between(rec_face, prec_face, alpha=0.2, color='red', step='post')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Face Detection PR Curve\nAP = {ap_face:.4f}', fontsize=13, fontweight='bold')
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

# Plate PR curve
ax = axes[1]
ax.step(rec_plate, prec_plate, 'b-', linewidth=2.5, where='post')
ax.fill_between(rec_plate, prec_plate, alpha=0.2, color='blue', step='post')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'License Plate PR Curve\nAP = {ap_plate:.4f}', fontsize=13, fontweight='bold')
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

# Combined view
ax = axes[2]
ax.step(rec_face, prec_face, 'r-', linewidth=2.5, label=f'Face (AP={ap_face:.3f})', where='post')
ax.step(rec_plate, prec_plate, 'b-', linewidth=2.5, label=f'Plate (AP={ap_plate:.3f})', where='post')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Both Classes\nmAP = {mAP:.4f}', fontsize=13, fontweight='bold')
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.suptitle('Precision-Recall Curves for Object Detection', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Reading a PR Curve:")
print("  - Top-left corner = perfect (precision=1, recall=1)")
print("  - Area under curve = AP")
print("  - A curve that stays high and to the right = a good model")
print("  - For Street View: we want HIGH RECALL even if precision dips")
print("    (blur everything suspicious rather than miss a face)")

## 5. Non-Maximum Suppression (NMS)

### The 12-Year-Old Version

When the model looks for faces, it gets really excited and draws MANY overlapping rectangles around the same face. It is like when you circle your answer on a test but accidentally circle it three times -- you only need one circle!

NMS is the "clean up crew." It:
1. Looks at the most confident box -- KEEPS it
2. Throws away any other box that overlaps too much with it (duplicate!)
3. Moves to the next most confident remaining box
4. Repeats until no boxes are left

### Staff-Level Detail

NMS is a greedy algorithm with O(N^2) complexity (comparing all pairs). Soft-NMS is a variant that decays confidence scores instead of hard removal, and is more robust for crowded scenes. NMS is typically run per-class.

In [ ]:
# === Non-Maximum Suppression: Step-by-Step Implementation ===

def nms_step_by_step(detections, iou_threshold=0.5, verbose=True):
    """
    Non-Maximum Suppression with detailed step-by-step output.
    
    Algorithm:
    1. Sort all boxes by confidence (highest first)
    2. Pick the top box -> KEEP it
    3. Compute IoU with all remaining boxes
    4. Remove any box with IoU > threshold (it is a duplicate)
    5. Repeat from step 2 with remaining boxes
    
    Time complexity: O(N^2) where N = number of detections
    Space complexity: O(N)
    """
    # Sort by confidence (highest first)
    sorted_dets = sorted(detections, key=lambda d: d['conf'], reverse=True)
    
    kept = []
    removed = []
    step = 0
    
    if verbose:
        print(f"=== Non-Maximum Suppression (IoU threshold = {iou_threshold}) ===")
        print(f"Starting with {len(sorted_dets)} detections\n")
    
    while sorted_dets:
        step += 1
        # Pick the best remaining detection
        best = sorted_dets.pop(0)
        kept.append(best)
        
        if verbose:
            print(f"--- Step {step} ---")
            print(f"  KEEP: conf={best['conf']:.2f}, bbox={best['bbox']}")
        
        # Check remaining detections
        remaining = []
        for det in sorted_dets:
            iou = compute_iou(best['bbox'], det['bbox'])
            if iou >= iou_threshold:
                if verbose:
                    print(f"  REMOVE: conf={det['conf']:.2f}, IoU={iou:.4f} >= {iou_threshold} (duplicate!)")
                removed.append(det)
            else:
                if verbose:
                    print(f"  KEEP:   conf={det['conf']:.2f}, IoU={iou:.4f} < {iou_threshold} (different object)")
                remaining.append(det)
        
        sorted_dets = remaining
        if verbose:
            print(f"  Remaining: {len(sorted_dets)} detections\n")
    
    if verbose:
        print(f"=== Result: {len(kept)} kept, {len(removed)} removed ===")
    
    return kept, removed


# === Example: Face Detection with Overlapping Boxes ===

# Simulate what Faster R-CNN outputs BEFORE NMS
raw_detections = [
    {"bbox": [100, 80, 60, 80], "conf": 0.95, "class": "face"},    # Best face 1 box
    {"bbox": [105, 85, 55, 75], "conf": 0.88, "class": "face"},    # Duplicate of face 1
    {"bbox": [98,  78, 62, 82], "conf": 0.82, "class": "face"},    # Another duplicate of face 1
    {"bbox": [300, 90, 50, 70], "conf": 0.91, "class": "face"},    # Best face 2 box
    {"bbox": [305, 95, 48, 68], "conf": 0.76, "class": "face"},    # Duplicate of face 2
    {"bbox": [200, 200, 80, 25], "conf": 0.87, "class": "plate"},  # License plate (no overlap)
]

kept, removed = nms_step_by_step(raw_detections, iou_threshold=0.5)

In [ ]:
# === Visualize NMS Before and After ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create synthetic image
img = np.zeros((300, 450, 3), dtype=np.uint8)
img[:120, :] = [135, 206, 235]
img[120:, :] = [100, 150, 100]
img[80:160, 100:160] = [220, 180, 150]  # Face 1
img[90:160, 300:350] = [200, 165, 135]  # Face 2
img[200:225, 200:280] = [240, 240, 240]  # Plate

# Before NMS
ax = axes[0]
ax.imshow(img)
for det in raw_detections:
    x, y, w, h = det['bbox']
    color = 'red' if det['class'] == 'face' else 'blue'
    rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor=color,
                              facecolor=color, alpha=0.15)
    ax.add_patch(rect)
    ax.text(x, y-3, f"{det['conf']:.2f}", color=color, fontsize=8, fontweight='bold')
ax.set_title(f'Before NMS: {len(raw_detections)} boxes\n(Many overlapping duplicates!)',
             fontsize=12, fontweight='bold')
ax.axis('off')

# After NMS
ax = axes[1]
ax.imshow(img)
for det in kept:
    x, y, w, h = det['bbox']
    color = 'red' if det['class'] == 'face' else 'blue'
    rect = patches.Rectangle((x, y), w, h, linewidth=2.5, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y-3, f"{det['class']} {det['conf']:.2f}", color=color,
            fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
ax.set_title(f'After NMS: {len(kept)} boxes\n(One box per object -- clean!)',
             fontsize=12, fontweight='bold')
ax.axis('off')

plt.suptitle('Non-Maximum Suppression in Action', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"NMS reduced {len(raw_detections)} boxes to {len(kept)} boxes.")
print(f"Removed {len(removed)} duplicate/overlapping detections.")
print()
print("NMS is one of the MOST COMMONLY asked algorithms in ML system design interviews.")
print("Be ready to write it on a whiteboard!")

## 6. Batch Prediction Pipeline

### The 12-Year-Old Version

Google has BILLIONS of Street View images. Processing them one at a time would take forever. Instead, they use an assembly line:

1. **CPU workers** prepare the images (resize, normalize) -- like prep cooks chopping vegetables
2. **GPU workers** run the detection model -- like chefs doing the actual cooking
3. **Blur workers** apply the blur effect -- like the plating team making it pretty

Each team works independently and passes their output to the next team. If the prep cooks are too slow, you hire more prep cooks without needing more chefs.

### Staff-Level Detail

The key architectural insight is **separating CPU-bound and GPU-bound workloads** into independent services. This allows:
- Independent horizontal scaling
- Better resource utilization (no idle GPUs waiting for CPU preprocessing)
- Fault isolation

In [ ]:
# === Batch Prediction Pipeline Architecture Diagram ===

from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Google Street View Blurring: Production Pipeline Architecture',
             fontsize=15, fontweight='bold', pad=20)

def draw_box(x, y, w, h, text, color, fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                         facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold')

def arrow(x1, y1, x2, y2, text=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))
    if text:
        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2
        ax.text(mid_x, mid_y + 0.15, text, ha='center', va='bottom',
                fontsize=8, style='italic', color='#666')

# === BATCH PREDICTION PIPELINE (top) ===
ax.text(8, 9.5, 'BATCH PREDICTION PIPELINE', ha='center', fontsize=13,
        fontweight='bold', color='#1565C0',
        bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.9))

draw_box(0.5, 7.5, 2.5, 1.2, 'Raw Street View\nImages\n(Object Storage)', '#FFE0B2')
draw_box(4.0, 7.5, 2.5, 1.2, 'Preprocessing\nService\n(CPU cluster)', '#B3E5FC')
draw_box(7.5, 7.5, 2.5, 1.2, 'Detection Model\n(Faster R-CNN)\n(GPU cluster)', '#C8E6C9')
draw_box(11.0, 7.5, 2.5, 1.2, 'NMS +\nBlurring\nService', '#F8BBD0')
draw_box(14.0, 7.5, 1.5, 1.2, 'Blurred\nImages', '#FFE0B2')

arrow(3.0, 8.1, 4.0, 8.1, 'images')
arrow(6.5, 8.1, 7.5, 8.1, 'tensors')
arrow(10.0, 8.1, 11.0, 8.1, 'bboxes')
arrow(13.5, 8.1, 14.0, 8.1)

# CPU/GPU labels
ax.text(5.25, 7.2, 'CPU', ha='center', fontsize=10, color='#0D47A1', fontweight='bold')
ax.text(8.75, 7.2, 'GPU', ha='center', fontsize=10, color='#1B5E20', fontweight='bold')

# === DATA/FEEDBACK PIPELINE (bottom) ===
ax.text(8, 5.0, 'DATA PIPELINE (Feedback Loop)', ha='center', fontsize=13,
        fontweight='bold', color='#C62828',
        bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.9))

draw_box(0.5, 3.0, 2.5, 1.2, 'User Reports\n(missed blurs)', '#FFCDD2')
draw_box(4.0, 3.0, 2.5, 1.2, 'Process\nReports', '#FFF9C4')
draw_box(7.5, 3.0, 2.5, 1.2, 'Hard Negative\nMining', '#D1C4E9')
draw_box(11.0, 3.0, 2.5, 1.2, 'Retrain\nModel', '#C8E6C9')
draw_box(14.0, 3.0, 1.5, 1.2, 'Updated\nModel', '#C8E6C9')

arrow(3.0, 3.6, 4.0, 3.6)
arrow(6.5, 3.6, 7.5, 3.6)
arrow(10.0, 3.6, 11.0, 3.6)
arrow(13.5, 3.6, 14.0, 3.6)

# Feedback arrow from updated model to detection model
ax.annotate('', xy=(8.75, 7.5), xytext=(14.75, 4.2),
            arrowprops=dict(arrowstyle='->', color='#C62828', lw=2, ls='--'))
ax.text(12.5, 5.8, 'deploy', fontsize=9, color='#C62828', fontweight='bold', rotation=45)

# Scaling annotations
ax.text(5.25, 6.5, 'Scale independently!\n(add more CPU nodes)', ha='center',
        fontsize=9, color='#0D47A1', style='italic')
ax.text(8.75, 6.5, 'Scale independently!\n(add more GPUs)', ha='center',
        fontsize=9, color='#1B5E20', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# === Simulate Batch Processing Pipeline ===

import time
from scipy.ndimage import gaussian_filter

class BatchPredictionPipeline:
    """
    Simulates the complete batch prediction pipeline.
    
    12-year-old version:
    An assembly line for blurring faces:
      Station 1 (CPU): Prepare the image
      Station 2 (GPU): Find faces and plates
      Station 3 (CPU): Remove duplicate boxes
      Station 4 (CPU): Blur the detected regions
    
    Staff-level detail:
    In production, each station is a microservice:
    - Preprocessing: Kubernetes pods with CPU instances
    - Detection: Kubernetes pods with GPU instances (e.g., NVIDIA T4)
    - Post-processing: CPU instances
    Communication via message queues (Kafka, PubSub)
    """
    
    def __init__(self, conf_threshold=0.3, nms_threshold=0.5):
        self.conf_threshold = conf_threshold
        self.nms_threshold = nms_threshold
    
    def preprocess(self, image, target_size=(800, 800)):
        """CPU-bound: resize and normalize."""
        normalized = image.astype(np.float32) / 255.0
        return normalized
    
    def detect(self, preprocessed_image):
        """GPU-bound: run Faster R-CNN (simulated here)."""
        return [
            {"bbox": [70, 90, 50, 60], "class": "face", "conf": 0.95},
            {"bbox": [75, 93, 48, 57], "class": "face", "conf": 0.80},  # duplicate
            {"bbox": [240, 110, 45, 55], "class": "face", "conf": 0.88},
            {"bbox": [310, 172, 55, 28], "class": "plate", "conf": 0.92},
        ]
    
    def apply_nms(self, detections):
        """CPU-bound: remove overlapping boxes."""
        sorted_dets = sorted(detections, key=lambda d: d['conf'], reverse=True)
        kept = []
        while sorted_dets:
            best = sorted_dets.pop(0)
            kept.append(best)
            sorted_dets = [
                d for d in sorted_dets
                if compute_iou(best['bbox'], d['bbox']) < self.nms_threshold
            ]
        return kept
    
    def blur_regions(self, image, detections, sigma=8):
        """CPU-bound: apply Gaussian blur to detected regions."""
        blurred = image.copy()
        for det in detections:
            x, y, w, h = [int(v) for v in det['bbox']]
            y_end = min(y + h, image.shape[0])
            x_end = min(x + w, image.shape[1])
            for c in range(3):
                blurred[max(0,y):y_end, max(0,x):x_end, c] = gaussian_filter(
                    blurred[max(0,y):y_end, max(0,x):x_end, c], sigma=sigma
                )
        return blurred
    
    def process_image(self, image):
        """Run the complete pipeline on one image."""
        preprocessed = self.preprocess(image)
        raw_detections = self.detect(preprocessed)
        filtered_detections = self.apply_nms(raw_detections)
        blurred_image = self.blur_regions(image, filtered_detections)
        return blurred_image, filtered_detections


# Run the pipeline
pipeline = BatchPredictionPipeline()

# Create synthetic image
img = np.zeros((300, 500, 3), dtype=np.uint8)
img[:150, :] = [135, 206, 235]
img[150:, :] = [128, 128, 128]
img[50:250, 50:150] = [180, 160, 140]
img[50:250, 200:350] = [160, 140, 130]
img[100:140, 80:110] = [220, 180, 150]
img[120:155, 250:275] = [200, 165, 135]
img[180:195, 320:360] = [200, 200, 200]

blurred, detections = pipeline.process_image(img)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(img)
axes[0].set_title('1. Original Image', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img)
for det in detections:
    x, y, w, h = det['bbox']
    color = 'red' if det['class'] == 'face' else 'blue'
    rect = patches.Rectangle((x, y), w, h, linewidth=2.5, edgecolor=color, facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x, y-3, f"{det['class']} {det['conf']:.0%}", color=color,
                 fontsize=9, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
axes[1].set_title(f'2. Detections (after NMS): {len(detections)} objects', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(blurred)
axes[2].set_title('3. Blurred Output (ready to serve)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Complete Blurring Pipeline: Input -> Detect -> Blur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Hard Negative Mining

### The 12-Year-Old Version

Imagine your model keeps confusing tree knots with faces. Every time it sees a round, dark spot on a tree, it thinks it is a face and blurs it. That is embarrassing!

Hard negative mining is like a tutor who says: "You keep getting this wrong. Let me give you EXTRA practice with tree knots until you learn they are NOT faces."

The process:
1. Run the model on images
2. Collect its MISTAKES (things it confidently thought were faces but were not)
3. Add those mistakes to the training data with the label "NOT a face"
4. Retrain the model
5. Now the model is better at avoiding those specific mistakes!

### Staff-Level Detail

Hard negatives are false positives with high confidence -- cases where the model is confidently wrong. These are the most informative examples for training because they represent the model's blind spots. In the feedback loop, user reports of missed blurs generate hard positives (faces the model missed), while false blurs generate hard negatives.

In [ ]:
# === Hard Negative Mining Implementation ===

def simulate_hard_negative_mining(model_predictions, ground_truths, 
                                   iou_threshold=0.3, conf_threshold=0.5):
    """
    Find hard negatives: confident false positives.
    
    12-year-old version:
    Find all the things the model was REALLY sure about but got WRONG.
    These are the model's worst mistakes -- the ones it needs to study.
    
    Staff-level detail:
    A hard negative is a prediction where:
    1. The model is confident (score > conf_threshold)
    2. It does NOT match any ground truth (IoU < iou_threshold with all GTs)
    
    These are added to the training set as negative examples with
    higher sampling weight to correct the model's blind spots.
    """
    hard_negatives = []
    true_positives = []
    easy_negatives = []
    
    for pred in model_predictions:
        # Check if this prediction matches any ground truth
        best_iou = max(
            (compute_iou(pred['bbox'], gt['bbox']) for gt in ground_truths),
            default=0
        )
        
        if best_iou >= iou_threshold:
            true_positives.append(pred)
        elif pred['conf'] >= conf_threshold:
            # High confidence + no match = HARD negative
            pred['best_iou'] = best_iou
            hard_negatives.append(pred)
        else:
            # Low confidence + no match = easy negative (model already uncertain)
            easy_negatives.append(pred)
    
    return hard_negatives, true_positives, easy_negatives


# Simulate model predictions on a batch of images
ground_truths = [
    {'bbox': [100, 80, 60, 80], 'class': 'face'},
    {'bbox': [300, 90, 50, 70], 'class': 'face'},
    {'bbox': [200, 200, 80, 25], 'class': 'plate'},
]

model_predictions = [
    {'bbox': [102, 82, 58, 78], 'conf': 0.95, 'class': 'face'},   # TP: matches GT face 1
    {'bbox': [298, 88, 52, 72], 'conf': 0.90, 'class': 'face'},   # TP: matches GT face 2
    {'bbox': [202, 198, 78, 27], 'conf': 0.88, 'class': 'plate'}, # TP: matches GT plate
    {'bbox': [50, 180, 40, 50], 'conf': 0.82, 'class': 'face'},   # HARD NEG: tree knot!
    {'bbox': [380, 120, 35, 35], 'conf': 0.75, 'class': 'face'},  # HARD NEG: round sign!
    {'bbox': [150, 250, 30, 30], 'conf': 0.65, 'class': 'face'},  # HARD NEG: ball!
    {'bbox': [420, 30, 20, 20], 'conf': 0.15, 'class': 'face'},   # Easy neg: model already unsure
    {'bbox': [10, 10, 15, 15], 'conf': 0.10, 'class': 'face'},    # Easy neg: model already unsure
]

hard_negs, tps, easy_negs = simulate_hard_negative_mining(
    model_predictions, ground_truths, iou_threshold=0.3, conf_threshold=0.5
)

print("=== Hard Negative Mining Results ===")
print(f"\n  True Positives (correct):     {len(tps)}")
for tp in tps:
    print(f"    conf={tp['conf']:.2f}, class={tp['class']}, bbox={tp['bbox']}")

print(f"\n  Hard Negatives (DANGER!):     {len(hard_negs)}")
for hn in hard_negs:
    print(f"    conf={hn['conf']:.2f}, class={hn['class']}, bbox={hn['bbox']}  <-- model was CONFIDENT but WRONG")

print(f"\n  Easy Negatives (no concern):  {len(easy_negs)}")
for en in easy_negs:
    print(f"    conf={en['conf']:.2f}, class={en['class']}, bbox={en['bbox']}  <-- model was already unsure")

print("\n" + "=" * 60)
print("ACTION PLAN:")
print(f"  1. Take {len(hard_negs)} hard negatives")
print(f"  2. Label them as 'NOT face' / 'NOT plate'")
print(f"  3. Add to training dataset with higher sampling weight")
print(f"  4. Retrain model -> fewer false alarms on these patterns")
print(f"  5. In production, user reports feed this pipeline continuously")

## 8. Scaling to Billions of Images

### The 12-Year-Old Version

Google has cars driving in every country, taking photos continuously. That is BILLIONS of images. How do you process all of them?

Imagine you need to check 1 billion homework assignments. You cannot do it alone! You hire thousands of helpers, give each one a stack of assignments, and they all work at the same time. That is **distributed batch processing**.

### Staff-Level Scaling Considerations

In [ ]:
# === Scaling Calculations ===

def compute_scaling_requirements(total_images, images_per_second_per_gpu, target_hours):
    """
    How many GPUs do we need to process all images within target time?
    
    This is the kind of back-of-envelope calculation interviewers love.
    """
    target_seconds = target_hours * 3600
    required_throughput = total_images / target_seconds  # images/second
    num_gpus = np.ceil(required_throughput / images_per_second_per_gpu)
    
    return {
        'total_images': total_images,
        'target_hours': target_hours,
        'required_throughput': required_throughput,
        'images_per_gpu': images_per_second_per_gpu,
        'num_gpus': int(num_gpus),
    }


print("=== Back-of-Envelope Scaling Calculations ===")
print("(The kind of math interviewers LOVE to see)\n")

# Faster R-CNN on NVIDIA V100: ~5 FPS (images per second)
fps_per_gpu = 5

scenarios = [
    (1_000_000, 24, "Initial dataset (1M images)"),
    (10_000_000, 24, "After augmentation (10M images)"),
    (100_000_000, 48, "Growing fleet (100M images)"),
    (1_000_000_000, 168, "Global coverage (1B images)"),
]

print(f"Assumption: Faster R-CNN @ {fps_per_gpu} FPS per GPU (NVIDIA V100)\n")
print(f"{'Scenario':<35} {'Images':>12} {'Target':>10} {'GPUs Needed':>12} {'Cost/hr':>10}")
print("-" * 85)

gpu_cost_per_hour = 3.0  # V100 spot instance

for total, hours, name in scenarios:
    result = compute_scaling_requirements(total, fps_per_gpu, hours)
    cost = result['num_gpus'] * gpu_cost_per_hour
    print(f"{name:<35} {total:>12,} {hours:>8}h {result['num_gpus']:>12,} ${cost:>9,.0f}")

print("\n=== Key Scaling Strategies ===")
strategies = [
    ("Batch GPU inference", "Process images in batches of 8-32 to maximize GPU utilization"),
    ("Mixed precision (FP16)", "2x faster inference with minimal accuracy loss"),
    ("Model distillation", "Train a smaller model that mimics the big one (3-5x speedup)"),
    ("Horizontal scaling", "Add more GPU nodes linearly (embarrassingly parallel)"),
    ("Priority queues", "Process new/high-traffic areas first"),
    ("Incremental processing", "Only reprocess when cameras recapture an area"),
]

for name, desc in strategies:
    print(f"  {name}: {desc}")

## 9. User Feedback Loop Architecture

### The Complete Picture

The system is never "done." Users find mistakes, those mistakes become training data, and the model gets better. This virtuous cycle is what makes production ML systems improve over time.

In [ ]:
# === User Feedback Loop Simulation ===

def simulate_feedback_loop(initial_recall, num_iterations=6):
    """
    Simulate how the model improves with user feedback over time.
    
    12-year-old version:
    Every time a user reports a missed face, the model learns from it.
    Over months, the model gets better and better.
    But it hits a ceiling -- some faces are just really hard!
    """
    recalls = [initial_recall]
    precisions = [0.70]  # Start with decent precision
    user_reports = []
    
    for i in range(num_iterations):
        # Each iteration: users report missed faces, model retrains
        missed_faces = int(10000 * (1 - recalls[-1]))  # Simulated reports
        user_reports.append(missed_faces)
        
        # Model improves (with diminishing returns)
        improvement = 0.03 * (1 - recalls[-1])  # Smaller improvements as we get better
        new_recall = min(0.999, recalls[-1] + improvement + np.random.normal(0, 0.005))
        
        # Precision might dip slightly as recall increases (trade-off)
        new_precision = precisions[-1] - 0.005 + np.random.normal(0, 0.008)
        new_precision = max(0.60, min(0.95, new_precision))
        
        recalls.append(new_recall)
        precisions.append(new_precision)
    
    return recalls, precisions, user_reports


np.random.seed(42)
recalls, precisions, reports = simulate_feedback_loop(initial_recall=0.92)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Recall improvement over time
ax = axes[0]
iterations = range(len(recalls))
ax.plot(iterations, recalls, 'g-o', linewidth=2.5, markersize=8)
ax.axhline(y=0.99, color='r', linestyle='--', alpha=0.5, label='Target: 99% recall')
ax.set_xlabel('Retraining Iteration', fontsize=12)
ax.set_ylabel('Recall', fontsize=12)
ax.set_title('Recall Improves with Feedback', fontsize=13, fontweight='bold')
ax.set_ylim([0.90, 1.005])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

# User reports decrease
ax = axes[1]
ax.bar(range(len(reports)), reports, color='#e74c3c', alpha=0.7)
ax.set_xlabel('Retraining Iteration', fontsize=12)
ax.set_ylabel('User Reports (missed faces)', fontsize=12)
ax.set_title('User Complaints Decrease', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Recall vs Precision trade-off
ax = axes[2]
colors = plt.cm.viridis(np.linspace(0, 1, len(recalls)))
for i in range(len(recalls)):
    ax.scatter(recalls[i], precisions[i], c=[colors[i]], s=100, zorder=3,
               edgecolors='black', linewidths=1)
    ax.annotate(f'v{i}', (recalls[i], precisions[i]), textcoords='offset points',
                xytext=(5, 5), fontsize=9)

# Draw arrow showing progression
for i in range(len(recalls)-1):
    ax.annotate('', xy=(recalls[i+1], precisions[i+1]),
                xytext=(recalls[i], precisions[i]),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Model Evolution\n(Moving toward high recall)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Continuous Improvement Through User Feedback', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("=== Feedback Loop Summary ===")
print(f"  Initial recall: {recalls[0]:.3f} -> Final recall: {recalls[-1]:.3f}")
print(f"  Improvement: {(recalls[-1] - recalls[0])*100:.1f} percentage points")
print(f"  Initial user reports: {reports[0]:,} -> Final: {reports[-1]:,}")
print(f"  Reduction: {(1 - reports[-1]/reports[0])*100:.0f}% fewer complaints")
print()
print("KEY INSIGHT: The model never stops improving.")
print("Every user report is a free training example.")
print("This is why production ML is a LOOP, not a one-time project.")

## Interview Cheat Sheet: Evaluation & Serving

### 30-Second Summary

"We evaluate the detection model using mAP, which averages AP across face and license plate classes. AP is computed by sorting predictions by confidence, matching to ground truths via IoU, and computing the area under the precision-recall curve. For serving, we use a batch prediction pipeline with separated CPU preprocessing and GPU inference for independent scaling. NMS removes duplicate detections post-inference. A user feedback loop with hard negative mining continuously improves the model by retraining on its worst mistakes."

### Key Metrics to Know

| Metric | What It Measures | When to Use |
|--------|------------------|-------------|
| **IoU** | Overlap between predicted and ground truth boxes | Basis for all detection metrics |
| **Precision@IoU** | Fraction of correct detections at a threshold | Too dependent on threshold choice |
| **AP** | Average precision across IoU thresholds (per class) | Per-class model quality |
| **mAP** | Average AP across all classes | Overall model quality (gold standard) |
| **User reports** | Real-world privacy violations | Online/production metric |

### NMS Pseudocode (Whiteboard Ready)

```
function NMS(detections, iou_threshold):
    sort detections by confidence (highest first)
    kept = []
    while detections is not empty:
        best = detections.pop(0)
        kept.append(best)
        detections = [d for d in detections 
                      if IoU(best, d) < iou_threshold]
    return kept
```

### Key Phrases to Drop

- "mAP averages AP across classes, giving a single quality number"
- "We separate CPU preprocessing from GPU inference for independent scaling"
- "NMS is greedy: keep the most confident, remove overlapping duplicates"
- "Hard negative mining focuses retraining on the model's worst mistakes"
- "The feedback loop means the system continuously improves from user reports"
- "For privacy, we tune the confidence threshold for HIGH RECALL, accepting some false positives"

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)